In [1]:
import arff
import pandas as pd

# загружаем данные
with open("c:/Personal/Science/Data/Binary_classification/KDDCup09_upselling.arff", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],
                  columns=[a[0] for a in dataset["attributes"]])
pd.set_option("display.max_columns", None)
df.head()

,Var6,Var13,Var21,Var22,Var24,Var25,Var28,Var35,Var38,Var57,Var65,Var73,Var74,Var76,Var78,Var81,Var83,Var85,Var109,Var112,Var113,Var119,Var123,Var125,Var126,Var132,Var133,Var134,Var140,Var144,Var149,Var153,Var160,Var163,Var196,Var203,Var205,Var207,Var208,Var210,Var211,Var218,Var221,Var223,Var227,UPSELLING
0,9723.0,908.0,4568.0,5710.0,54.0,840.0,287.76,0.0,3712122.0,2.320231,9.0,32.0,105.0,2663904.0,0.0,16084.53,590.0,102.0,1232.0,1736.0,67110.00,11350.0,1608.0,12465.0,-20.0,0.0,3666460.0,901128.0,5065.0,0.0,1140734.0,10151040.0,766.0,695202.0,0,0,1,10,0,5,1,cJvF,4,0,2,-1
1,840.0,3108.0,212.0,265.0,6.0,88.0,166.56,0.0,15900.0,1.325785,27.0,126.0,343.0,324960.0,0.0,14406.96,0.0,8.0,24.0,72.0,10296.44,690.0,6.0,92097.0,32.0,0.0,543125.0,160770.0,3110.0,18.0,230930.0,1064396.0,56.0,202914.0,0,0,1,9,0,5,0,UYBR,6,0,1,-1
2,259.0,0.0,128.0,160.0,6.0,128.0,200.00,0.0,0.0,6.782952,9.0,18.0,0.0,795360.0,0.0,30113.10,0.0,4.0,56.0,72.0,4948.44,460.0,0.0,0.0,4.0,0.0,270145.0,280598.0,0.0,9.0,201691.0,1221824.0,10.0,70248.0,0,0,1,10,0,5,0,cJvF,4,0,2,-1
3,630.0,596.0,24.0,30.0,0.0,0.0,186.64,0.0,5485038.0,0.371502,9.0,100.0,231.0,2779944.0,0.0,194665.20,10.0,0.0,24.0,8.0,-64855.20,230.0,18.0,28665.0,-30.0,0.0,0.0,0.0,415.0,27.0,0.0,9885080.0,0.0,0.0,0,0,2,10,0,5,0,UYBR,4,3,2,-1
4,140.0,0.0,64.0,80.0,0.0,24.0,220.08,0.0,4279404.0,3.092074,9.0,16.0,0.0,0.0,0.0,124983.90,5.0,4.0,0.0,0.0,451612.00,460.0,18.0,0.0,-30.0,0.0,5866200.0,804074.0,0.0,9.0,0.0,9355360.0,14.0,273504.0,0,0,1,10,0,5,0,UYBR,4,0,2,-1


In [5]:
from typing import List, Any
import numpy as np

def clean_dataset(
    df: pd.DataFrame,
    cols_to_drop: List[Any] = None,
    missing_values: List[Any] = None
) -> pd.DataFrame:
    """
    Предобработка датафрейма с учетом пользовательских меток пропусков:
    1. Заполняет пропущенные значения
       - Модой для категориальных/текстовых признаков.
       - Средним значением для числовых признаков.
    2. Удаляет признаки, у которых одно значение занимает 100% строк.
    """
    df_cleaned = df.copy()
    n_rows = len(df_cleaned)
    
    if n_rows == 0:
        return df_cleaned

    # Задаем список меток для пропущенных значений по умолчанию, если не передан свой
    if missing_values is None:
        missing_values = ['', ' ', 'N/A', 'NA', 'null', 'nan', 'NaN', 'None', 'none', '?', '-']

    # 1. Приведение пользовательских пропусков к np.nan и их импутация
    for col in df_cleaned.columns:
        # Заменяем все кастомные метки пропусков на np.nan
        df_cleaned[col] = df_cleaned[col].replace(missing_values, np.nan)

        if df_cleaned[col].isna().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_cleaned[col]):
                # а) Числовые переменные -> средним значением
                mean_val = df_cleaned[col].mean()
                df_cleaned[col] = df_cleaned[col].fillna(mean_val)
            else:
                # б) Категориальные переменные -> модой
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(mode_val[0])

    # 2. Проверка условий для удаления колонок
    for col in df_cleaned.columns:
        series = df_cleaned[col]
        max_value_count = series.value_counts().iloc[0]
        max_value_ratio = max_value_count / n_rows

        # Если одинаковых значений = 100% -> к удалению
        if max_value_ratio == 1:
            cols_to_drop.append(col)

    # Удаляем выявленные колонки
    if cols_to_drop:
        df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    return df_cleaned

df = clean_dataset(df)

In [7]:
df['UPSELLING'] = df['UPSELLING'].astype(int)
df['UPSELLING'] = df['UPSELLING'].replace(-1,0)
y = df["UPSELLING"]
X = df.drop(["UPSELLING"], axis=1)

In [9]:
import time
import category_encoders as ce
from skrub import SimilarityEncoder, MinHashEncoder
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from k_fold_encoder import KFoldTargetEncoder
from k_fold_glmm_encoder import KFoldGLMMEncoder

cat_cols = ['Var196', 'Var203', 'Var205', 'Var207', 'Var208', 'Var210', 'Var211',
                    'Var218', 'Var221', 'Var223', 'Var227']

encoders = {
    "MinHash": lambda: ColumnTransformer([(
                f"minhash_{col}",
                MinHashEncoder(),
                [col]
            )
            for col in cat_cols
        ],
        remainder="passthrough"
    ),
    "Similarity": lambda: ColumnTransformer([
        ("cat", SimilarityEncoder(), cat_cols)], 
        remainder="passthrough"),
   'Leave-one-out': lambda: ce.LeaveOneOutEncoder(cols=cat_cols),
   'K-fold':        lambda: KFoldTargetEncoder(cols=cat_cols),
   "CatBoost":      lambda: ce.CatBoostEncoder(cols=cat_cols),
   "CV-GLMM":       lambda: KFoldGLMMEncoder(cols=cat_cols),
   "WOE":           lambda: ce.WOEEncoder(cols=cat_cols),
   "JS":            lambda: ce.JamesSteinEncoder(cols=cat_cols),
   "MT":            lambda: ce.TargetEncoder(cols=cat_cols),
   'GLMM':          lambda: ce.GLMMEncoder(cols=cat_cols),
   'M-Estimate':    lambda: ce.MEstimateEncoder(cols=cat_cols),
   "OneHot":  lambda: ce.OneHotEncoder(cols=cat_cols),
   "Gray":    lambda: ce.GrayEncoder(cols=cat_cols),
   "BaseN":   lambda: ce.BaseNEncoder(cols=cat_cols),
   "Hashing": lambda: ce.HashingEncoder(cols=cat_cols,n_components=32),
   "Helmert": lambda: ce.HelmertEncoder(cols=cat_cols),
   "Sum":     lambda: ce.SumEncoder(cols=cat_cols),
   "BackwardDifference": lambda: ce.BackwardDifferenceEncoder(cols=cat_cols)
}

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():
    for encoder_name, encoder in encoders.items():
        
        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            enc_instance = encoder()  # Вызываем lambda, получая объект кодировщика
            X_train_encoded = enc_instance.fit_transform(X_train, y_train)
            X_test_encoded = enc_instance.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Empirical",
                    "dataset": "KDDCup09",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Empirical, KDDCup09, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_KDDCup09_empirical.csv", index=True)
print("Done!")

Empirical, KDDCup09, LogReg, MinHash, 1, 1, 0.8284065448578372, 0.8294256872765106, 0.7269116186693148, 0.7301275760549558, 0.7209302325581395, 0.7395626242544732, 0.41236209869384766
Empirical, KDDCup09, LogReg, MinHash, 1, 2, 0.8531036006185113, 0.8542588085604035, 0.76266137040715, 0.7659157688540646, 0.7562862669245648, 0.7757936507936508, 0.35030603408813477
Empirical, KDDCup09, LogReg, MinHash, 1, 3, 0.8350730606421116, 0.8403925194169182, 0.7485089463220675, 0.7555555555555555, 0.7349624060150376, 0.7773359840954275, 0.3222794532775879
Empirical, KDDCup09, LogReg, MinHash, 1, 4, 0.8306621503582878, 0.8304942238558916, 0.7504970178926441, 0.7527093596059113, 0.74609375, 0.7594433399602386, 0.2847597599029541
Empirical, KDDCup09, LogReg, MinHash, 1, 5, 0.8465469607800513, 0.848813917242403, 0.7564612326043738, 0.757185332011893, 0.7549407114624506, 0.7594433399602386, 0.32846593856811523
Empirical, KDDCup09, LogReg, MinHash, 2, 1, 0.8525947489665184, 0.8581881479652343, 0.75670307

C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, KDDCup09, CART, Leave-one-out, 2, 1, 0.5029860519423144, 0.5016384879825883, 0.5034756703078451, 0.0196078431372549, 0.7142857142857143, 0.009940357852882704, 0.1182548999786377
Empirical, KDDCup09, CART, Leave-one-out, 2, 2, 0.5, 0.5004965243296922, 0.49950347567030784, 0.0, 0.0, 0.0, 0.11621928215026855


C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, KDDCup09, CART, Leave-one-out, 2, 3, 0.5029821073558648, 0.5021300766827606, 0.5029821073558648, 0.0196078431372549, 0.7142857142857143, 0.009940357852882704, 0.11196064949035645
Empirical, KDDCup09, CART, Leave-one-out, 2, 4, 0.49801192842942343, 0.5, 0.49801192842942343, 0.0, 0.0, 0.0, 0.15790462493896484
Empirical, KDDCup09, CART, Leave-one-out, 2, 5, 0.49801192842942343, 0.5, 0.49801192842942343, 0.0, 0.0, 0.0, 0.15923118591308594
Empirical, KDDCup09, CART, Leave-one-out, 3, 1, 0.4990118810943861, 0.4991078356080004, 0.49950347567030784, 0.007874015748031496, 0.4, 0.003976143141153081, 0.12663793563842773
Empirical, KDDCup09, CART, Leave-one-out, 3, 2, 0.5009861466123892, 0.5010594770844452, 0.5004965243296922, 0.015655577299412915, 0.5714285714285714, 0.007936507936507936, 0.1426982879638672
Empirical, KDDCup09, CART, Leave-one-out, 3, 3, 0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.13946104049682617


C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


... (107 outputs left out)

C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, KDDCup09, LightGBM, Leave-one-out, 4, 3, 0.7365034445414986, 0.766025293885275, 0.5, 0.0, 0.0, 0.0, 0.32270216941833496


C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, KDDCup09, LightGBM, Leave-one-out, 4, 4, 0.5228055128473691, 0.5329208355544169, 0.5, 0.0, 0.0, 0.0, 0.31031274795532227
Empirical, KDDCup09, LightGBM, Leave-one-out, 4, 5, 0.6366611464414309, 0.5738645353413518, 0.5159045725646123, 0.6724949562878278, 0.508130081300813, 0.9940357852882704, 0.3464672565460205
Empirical, KDDCup09, LightGBM, Leave-one-out, 5, 1, 0.6652801445296473, 0.6006542501479282, 0.5094339622641509, 0.6702269692923899, 0.5045226130653266, 0.9980119284294234, 0.37184739112854004
Empirical, KDDCup09, LightGBM, Leave-one-out, 5, 2, 0.7915601628325287, 0.744116809571441, 0.522343594836147, 0.6769644056413701, 0.5116751269035533, 1.0, 0.3365330696105957
Empirical, KDDCup09, LightGBM, Leave-one-out, 5, 3, 0.5406032986968843, 0.5393585773948704, 0.5, 0.007889546351084813, 0.5, 0.003976143141153081, 0.341841459274292
Empirical, KDDCup09, LightGBM, Leave-one-out, 5, 4, 0.6601445007885095, 0.6245234189510365, 0.49801192842942343, 0.0, 0.0, 0.0, 0.3129580020904541
E